# Homework 3

## 1. Time Series Classification Part 1: Feature Creation/Extraction

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### (a) and (b)

In [5]:
DATA_PATH = Path("AReM")

activity_folders = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

In [6]:
column_names = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

In [7]:
def read_dataset(file_path):
    file_path = Path(file_path)

    if file_path.parent.name == "bending2" and file_path.name == "dataset4.csv":
        data = pd.read_csv(
            file_path,
            comment="#",
            sep=r"\s+",
            header=None,
            names=column_names
        )
    else:
        data = pd.read_csv(
            file_path,
            comment="#",
            header=None,
            usecols=range(7),
            names=column_names
        )
    data = data.apply(pd.to_numeric, errors="coerce")
    data = data.sort_values("time")

    if len(data) < 480:
        expected_time = np.arange(0, 120000, 250)

        data = (
            data.set_index("time")
                .reindex(expected_time)
                .interpolate(method="linear")
                .reset_index()
                .rename(columns={"index": "time"})
        )

    return data

In [10]:
train_files = []
test_files = []

for activity in activity_folders:
    folder_path = DATA_PATH / activity

    for file_path in folder_path.glob("dataset*.csv"):
        dataset_number = int(file_path.stem.replace("dataset", ""))

        if activity in ["bending1", "bending2"]:
            is_test = dataset_number in [1, 2]
        else:
            is_test = dataset_number in [1, 2, 3]

        file_info = {
            "activity": activity,
            "dataset": dataset_number,
            "file_path": file_path
        }

        if is_test:
            test_files.append(file_info)
        else:
            train_files.append(file_info)
            
print("Number of training instances:", len(train_files))
print("Number of test instances:", len(test_files))
print("Total:", len(train_files) + len(test_files))

Number of training instances: 69
Number of test instances: 19
Total: 88


### (c)

#### i

Include minimum, maximum, mean, median, standard deviation, variance, range, first quartile, third quartile, interquartile range, root mean square, skewness, kurtosis, and zero-crossing rate.

#### ii

In [12]:
time_series_columns = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

def extract_features(data):
    features = {}

    for i, column in enumerate(time_series_columns, start=1):
        series = data[column]

        features[f"min{i}"] = series.min()
        features[f"max{i}"] = series.max()
        features[f"mean{i}"] = series.mean()
        features[f"median{i}"] = series.median()
        features[f"std{i}"] = series.std()
        features[f"first_quartile{i}"] = series.quantile(0.25)
        features[f"third_quartile{i}"] = series.quantile(0.75)

    return features

In [13]:
def create_feature_dataset(file_list, set_name):
    rows = []

    for file_info in file_list:
        data = read_dataset(file_info["file_path"])
        features = extract_features(data)

        features["activity"] = file_info["activity"]
        features["dataset"] = file_info["dataset"]
        features["set"] = set_name

        rows.append(features)

    return pd.DataFrame(rows)

train_features = create_feature_dataset(train_files, "train")
test_features = create_feature_dataset(test_files, "test")

feature_dataset = pd.concat(
    [train_features, test_features],
    ignore_index=True
)

In [15]:
feature_columns = []

for i in range(1, 7):
    feature_columns.extend([
        f"min{i}",
        f"max{i}",
        f"mean{i}",
        f"median{i}",
        f"std{i}",
        f"first_quartile{i}",
        f"third_quartile{i}"
    ])

feature_dataset = feature_dataset[
    ["activity", "dataset", "set"] + feature_columns
]

print("Shape of feature dataset:", feature_dataset.shape)
feature_dataset.head()
print("Number of extracted features:", len(feature_columns))

Shape of feature dataset: (88, 45)
Number of extracted features: 42


#### iii

In [17]:
def bootstrap_std_confidence_intervals(
    data,
    n_bootstrap=10000,
    confidence_level=0.90,
    random_state=42
):
    rng = np.random.default_rng(random_state)
    values = data.to_numpy()

    n_instances, n_features = values.shape

    bootstrap_std = np.empty((n_bootstrap, n_features))

    for i in range(n_bootstrap):
        sample_indices = rng.choice(
            n_instances,
            size=n_instances,
            replace=True
        )

        bootstrap_sample = values[sample_indices]

        bootstrap_std[i] = np.std(
            bootstrap_sample,
            axis=0,
            ddof=1
        )

    alpha = 1 - confidence_level

    lower_bounds = np.quantile(
        bootstrap_std,
        alpha / 2,
        axis=0
    )

    upper_bounds = np.quantile(
        bootstrap_std,
        1 - alpha / 2,
        axis=0
    )

    original_std = data.std(ddof=1)

    results = pd.DataFrame({
        "Feature": data.columns,
        "Standard Deviation": original_std.values,
        "90% CI Lower": lower_bounds,
        "90% CI Upper": upper_bounds
    })

    return results

bootstrap_results = bootstrap_std_confidence_intervals(
    feature_dataset[feature_columns],
    n_bootstrap=10000,
    confidence_level=0.90,
    random_state=42
)
bootstrap_results.round(3)

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.570,8.244,10.759
1,max1,4.394,3.327,5.306
2,mean1,5.336,4.706,5.879
3,median1,5.440,4.796,5.994
4,std1,1.772,1.573,1.949
5,first_quartile1,6.154,5.569,6.639
6,third_quartile1,5.139,4.327,5.827
7,min2,0.000,0.000,0.000
8,max2,5.063,4.618,5.410
9,mean2,1.574,1.394,1.706


#### iv

I selected minimum, mean, and maximum. The minimum and maximum can describe the range and extreme behavior, and the mean represents overall level.

## 2. ISLR 3.7.4

### (a)

The training RSS of cubic regression will be less than or equal to that of linear regression. In linear model, the coefficients of \(X^2\) and \(X^3\) are zero, that means linear model is a special case of the cubic model. So whether it's a linear relationship or not, a cubic model can fit it.

### (b)

The test RSS of linear regression will be lower. Because the relationship of X and Y is linear, the cubic terms will not reduce bias. It can increase variance by fitting noise, which can lead to overfit.

### (c)

The training RSS of cubic regression will be less than or equal to that of linear regression. The reason is same as the questain (a).

### (d)

Unable to determine. The cubic model havs lower bias and higher variance. If the relationship is strongly nonlinear, the cubic model may perform better. If the relationship is slightly nonlinear, the linear model may perform better.